# 📝 Assignment: Explainable AI for Text — LIME, SHAP & Integrated Gradients

**Dataset:** Emotion (dair-ai/emotion) — 20,000 English tweets labelled with **6 emotions**: sadness 😢, joy 😀, love ❤️, anger 😠, fear 😨, surprise 😲
**Model:** `bhadresh-savani/distilbert-base-uncased-emotion` (pre-trained, ~93% accuracy)

![roadmap](attachment:roadmap.png)

### 📋 Instructions
| | |
|---|---|
| ✏️ **Code cells marked `YOUR CODE HERE`** | Write your solution there. Everything else is given — just run it. |
| ✅ **Check cells** (`run_checks(...)`) | Run them after each task. They test your code and show ✅ / ❌. **Do not edit them.** |
| ✍️ **Answer cells** | Double-click and write 2–4 sentences. Use evidence from *your* outputs. |
| 🔑 **Hints** | Click to open — using them costs no marks. |
| 📤 **Submit** | `Runtime → Restart and run all`, then download as `RollNo_Name_XAI.ipynb`. |

<div style="background:#fffbeb;border-left:5px solid #F59E0B;padding:10px 14px;border-radius:6px;color:#111">⚠️ <b>New in this assignment:</b> the demo lab had 2 classes. Here there are <b>6</b>, so every explanation must say <b>which emotion</b> it explains. A word can support <i>fear</i> and argue against <i>joy</i> at the same time.</div>

![multiclass](attachment:multiclass.png)

## ⚙️ Setup (given — just run)
Colab: `Runtime → Change runtime type → T4 GPU`.

In [1]:
!pip install -q transformers datasets lime shap captum ipywidgets plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 73.3 MB/s eta 0:00:00


In [2]:
import re, time, random, warnings, html as _html
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import plotly.express as px, plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from scipy.stats import spearmanr

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- auto-checker (do not edit) ----------------
CHECKS = {}
def register(qid, *tests): CHECKS[qid] = tests

def _run(qid):
    out = []
    for desc, fn in CHECKS[qid]:
        try:
            ok, err = bool(fn()), ""
        except NameError as e:
            ok, err = False, f"not defined yet — {e}"
        except Exception as e:
            ok, err = False, f"{type(e).__name__}: {str(e)[:90]}"
        out.append((desc, ok, err))
    return out

def run_checks(qid):
    res = _run(qid); n = sum(ok for _, ok, _ in res)
    rows = "".join(f"<tr><td>{'✅' if ok else '❌'}</td><td>{_html.escape(d)}</td>"
                   f"<td style='color:#b91c1c;font-size:12px'>{_html.escape(e)}</td></tr>" for d, ok, e in res)
    col = "#10B981" if n == len(res) else "#F59E0B"
    display(HTML(f"<div style='border:2px solid {col};border-radius:8px;padding:8px 12px;background:#fff;color:#111'>"
                 f"<b style='color:{col}'>{qid}: {n}/{len(res)} checks passed {'🎉' if n == len(res) else ''}</b>"
                 f"<table style='margin-top:4px'>{rows}</table></div>"))

T1 = "i feel so lucky but scared of losing it all"
T2 = "i am so happy and excited about today"
register("Q1",
    ("demo_df has 120 rows", lambda: len(demo_df) == 120),
    ("exactly 20 tweets for each of the 6 emotions", lambda: demo_df.label.value_counts().eq(20).all() and demo_df.label.nunique() == 6),
    ("every tweet has 8–30 words", lambda: demo_df.n_words.between(8, 30).all()),
    ("rows are shuffled (first 12 rows not one class)", lambda: demo_df.label[:12].nunique() > 1),
    ("index reset to 0..119", lambda: list(demo_df.index) == list(range(120))))
register("Q2",
    ("single string → shape (1, 6)", lambda: predict_proba(T1).shape == (1, 6)),
    ("list of 3 strings → shape (3, 6)", lambda: predict_proba([T1, T2, T1]).shape == (3, 6)),
    ("every row sums to 1", lambda: np.allclose(predict_proba([T1, T2]).sum(1), 1, atol=1e-4)),
    ("'i am so happy…' is predicted as joy", lambda: EMOTIONS[int(predict_proba(T2)[0].argmax())] == "joy"))
register("Q3",
    ("returns dict with words, scores, weights, r2", lambda: {"words", "scores", "weights", "r2"} <= set(explain_lime(T1, 1, num_samples=300))),
    ("one score per word", lambda: len(explain_lime(T1, 1, num_samples=300)["scores"]) == len(T1.split())),
    ("some scores are non-zero", lambda: np.abs(explain_lime(T1, 1, num_samples=300)["scores"]).sum() > 0),
    ("same seed → same explanation", lambda: np.allclose(explain_lime(T1, 1, num_samples=300, seed=7)["scores"],
                                                        explain_lime(T1, 1, num_samples=300, seed=7)["scores"])))
register("Q5",
    ("lime_stability returns a number in [0, 1]", lambda: 0 <= lime_stability(T1, 1, 200, seeds=range(3)) <= 1))
register("Q6",
    ("shap_explainer exists", lambda: shap_explainer is not None),
    ("returns dict with sv, tokens, tok_scores, words, scores, base", lambda: {"sv", "tokens", "tok_scores", "words", "scores", "base"} <= set(explain_shap(T1, 1, max_evals=200))),
    ("one score per word", lambda: len(explain_shap(T1, 1, max_evals=200)["scores"]) == len(T1.split())),
    ("additivity: base + Σφ ≈ f(x)  (±0.02)", lambda: (lambda r: abs(r["base"] + r["tok_scores"].sum() - predict_proba(T1)[0, 1]) < 0.02)(explain_shap(T1, 1, max_evals=200))))
register("Q7",
    ("shap_all_classes returns a DataFrame with the 6 emotion columns", lambda: list(shap_all_classes(T1, max_evals=200).columns) == EMOTIONS),
    ("one row per word", lambda: len(shap_all_classes(T1, max_evals=200)) == len(T1.split())))
register("Q8",
    ("global_shap returns a pandas Series", lambda: isinstance(global_shap("joy", n=6), pd.Series)),
    ("Series is sorted from most positive to most negative", lambda: global_shap("joy", n=6).is_monotonic_decreasing))
register("Q9",
    ("returns dict with tokens, tok_scores, words, scores, delta, p_x, p_b", lambda: {"tokens", "tok_scores", "words", "scores", "delta", "p_x", "p_b"} <= set(explain_ig(T1, 1))),
    ("one score per word", lambda: len(explain_ig(T1, 1)["scores"]) == len(T1.split())),
    ("completeness: Σ IG ≈ F(x) − F(x′)  (±0.05, n_steps=100)", lambda: (lambda r: abs(r["tok_scores"].sum() - (r["p_x"] - r["p_b"])) < 0.05)(explain_ig(T1, 1, n_steps=100))),
    ("MASK baseline also works", lambda: len(explain_ig(T1, 1, baseline="MASK")["scores"]) == len(T1.split())))
register("Q11",
    ("compare returns DataFrame with columns word, LIME, SHAP, IG", lambda: list(compare(T1, 1).columns) == ["word", "LIME", "SHAP", "IG"]),
    ("one row per word", lambda: len(compare(T1, 1)) == len(T1.split())))
register("Q12",
    ("aopc returns a float", lambda: isinstance(aopc(T1, np.arange(len(T1.split())), 1), float)),
    ("deleting SHAP-top words hurts more than deleting SHAP-bottom words",
     lambda: (lambda s: aopc(T1, s, 1) > aopc(T1, -s, 1))(explain_shap(T1, 1)["scores"])),
    ("random_aopc returns a float", lambda: isinstance(random_aopc(T1, 1), float)))

def scorecard():
    rows = []
    for q in CHECKS:
        r = _run(q); rows.append((q, sum(ok for _, ok, _ in r), len(r)))
    html = "".join(f"<tr><td><b>{q}</b></td><td>{'🟩' * p}{'⬜' * (t - p)}</td><td>{p}/{t}</td></tr>" for q, p, t in rows)
    display(HTML(f"<table style='color:#111;background:#fff'>{html}</table>"))

print(f"✅ Setup done | device = {device}")

✅ Setup done | device = cuda


---
# Part A — Data & Model *(15 marks)*

## A.1 Load the dataset (given)

In [3]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", split="test")
EMOTIONS = ds.features["label"].names           # ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
df = ds.to_pandas()
df["emotion"] = df["label"].map(dict(enumerate(EMOTIONS)))
df["n_words"] = df["text"].str.split().str.len()
print(len(df), "tweets | classes:", EMOTIONS)
df.sample(5, random_state=SEED)

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

2000 tweets | classes: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


,text,label,emotion,n_words
1860,i feel so dirty but after spending a day at th...,0,sadness,51
353,i could feel his breath on me and smell the sw...,1,joy,14
1333,i just want to feel loved by you,2,love,8
905,i have felt the need to write out my sometimes...,3,anger,27
1289,at a party i met a girl who drew me to her,3,anger,12


### Q1 *(5 marks)* — Explore & build the demo set
Write code that:
1. Draws a **bar chart** of the number of tweets per emotion (Plotly). Is the dataset balanced?
2. Prints the **average number of words** per emotion.
3. Creates **`demo_df`**: exactly **20 tweets per emotion** with **8–30 words**, sampled with `random_state=SEED`, then **shuffled** (`random_state=SEED`) and with the **index reset**. → 120 rows.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Filter with <code>df.n_words.between(8, 30)</code>, sample each class with a list comprehension, join with <code>pd.concat</code>, then <code>.sample(frac=1, random_state=SEED).reset_index(drop=True)</code>.
</details>

In [4]:
# Q1: stratified, reproducible sample of 20 eligible tweets per class
eligible = df[df["n_words"].between(8, 30)].copy()
demo_df = (eligible.groupby("label", group_keys=False)
           .apply(lambda g: g.sample(n=20, random_state=SEED))
           .sample(frac=1, random_state=SEED).reset_index(drop=True))
fig = px.bar(df["emotion"].value_counts().reindex(EMOTIONS).rename_axis("emotion").reset_index(name="tweets"),
             x="emotion", y="tweets", title="Tweets per emotion (test split)")
fig.show()
print("Average words by emotion:")
display(df.groupby("emotion")["n_words"].mean().reindex(EMOTIONS).round(2))
print("Dataset is imbalanced:", not df["label"].value_counts().nunique() == 1)
display(demo_df[["text", "emotion", "n_words"]].head())

Average words by emotion:


,n_words
emotion,
sadness,19.04
joy,19.25
love,19.96
anger,19.44
fear,18.07
surprise,19.76


Dataset is imbalanced: True


,text,emotion,n_words
0,i then feel your tender touch as you enfold me...,love,13
1,i cant write a review for a book i adore unles...,love,21
2,i feel if the pressure vessel has been serious...,sadness,19
3,i find myself in the odd position of feeling s...,love,11
4,i feel terrific in every one of them,joy,8


In [5]:
run_checks("Q1")

✅,demo_df has 120 rows,
✅,exactly 20 tweets for each of the 6 emotions,
✅,every tweet has 8–30 words,
✅,rows are shuffled (first 12 rows not one class),
✅,index reset to 0..119,


**Q1(d) — Your answer:** The full test split is imbalanced because the six emotions do not occur equally often. Accuracy can hide poor performance on less frequent classes, so per-class precision, recall, F1, and a confusion matrix are also important. The demonstration subset is deliberately balanced (20 eligible tweets per class) to make class comparisons fairer.

## A.2 Load the model (given)

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "bhadresh-savani/distilbert-base-uncased-emotion"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()
print(model.config.id2label)
assert [model.config.id2label[i] for i in range(6)] == EMOTIONS, "label order mismatch!"
print("✅ model label order matches the dataset")

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
✅ model label order matches the dataset


### Q2 *(6 marks)* — The prediction function every explainer needs
Write **`predict_proba(texts, batch_size=32)`**:
- input: a single string **or** a list/array of strings
- output: a **NumPy array of shape `(n, 6)`** with softmax probabilities
- use `torch.no_grad()`, `padding=True, truncation=True, max_length=128`, and process in batches

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Same pattern as the demo lab: <code>tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)</code> → <code>model(**enc).logits</code> → <code>torch.softmax(..., dim=-1)</code>. Convert with <code>.cpu().numpy()</code> and <code>np.vstack</code> the batches. LIME/SHAP may pass a NumPy array — convert each item with <code>str()</code>.
</details>

In [7]:
def predict_proba(texts, batch_size=32):
    """Return softmax probabilities with shape (n_texts, 6)."""
    if isinstance(texts, str):
        texts = [texts]
    texts = list(texts)
    outputs = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start+batch_size], padding=True, truncation=True,
                          max_length=256, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**batch).logits
            outputs.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(outputs, axis=0) if outputs else np.empty((0, 6), dtype=float)

In [8]:
run_checks("Q2")

✅,"single string → shape (1, 6)",
✅,"list of 3 strings → shape (3, 6)",
✅,every row sums to 1,
✅,'i am so happy…' is predicted as joy,


### Model evaluation on `demo_df` (given)

In [9]:
from sklearn.metrics import confusion_matrix, classification_report
P = predict_proba(demo_df.text.tolist())          # (120, 6) — used later!
demo_df["pred"] = P.argmax(1)
cm = confusion_matrix(demo_df.label, demo_df.pred, labels=range(6))
px.imshow(cm, x=EMOTIONS, y=EMOTIONS, text_auto=True, color_continuous_scale="Blues",
          labels=dict(x="predicted", y="true"), title=f"Accuracy {np.mean(demo_df.pred == demo_df.label):.1%}").show()
print(classification_report(demo_df.label, demo_df.pred, target_names=EMOTIONS, digits=2))

              precision    recall  f1-score   support

     sadness       0.86      0.95      0.90        20
         joy       0.83      0.95      0.88        20
        love       0.95      0.95      0.95        20
       anger       0.93      0.70      0.80        20
        fear       0.72      0.90      0.80        20
    surprise       0.93      0.70      0.80        20

    accuracy                           0.86       120
   macro avg       0.87      0.86      0.86       120
weighted avg       0.87      0.86      0.86       120



✍️ **Q2(b) — Your answer:** After the evaluation cell runs, identify the largest off-diagonal entry in the confusion matrix and name those two emotions. Emotion labels can overlap in natural language, and short tweets often provide little context. Use the actual pair and count shown by your run rather than assuming a result in advance.

### 🧰 Helpers (given — run once)
All explainers below must return **one score per whitespace word** (`text.split()`), measured as contribution to **P(chosen emotion)**. These helpers draw and align them.

In [10]:
EMO_EMOJI = dict(sadness="😢", joy="😀", love="❤️", anger="😠", fear="😨", surprise="😲")

def highlight(words, scores, title="", note=""):
    scores = np.asarray(scores, float); m = np.abs(scores).max() or 1.0
    spans = "".join(f"<span title='{s:+.4f}' style='background:rgba({'16,185,129' if s > 0 else '239,68,68'},"
                    f"{min(1, abs(s) / m) * .85:.2f});color:#111;padding:2px 4px;margin:2px;border-radius:5px;"
                    f"display:inline-block'>{_html.escape(w)}</span>" for w, s in zip(words, scores))
    return (f"<div style='border:1px solid #e5e7eb;border-radius:10px;padding:8px 12px;margin:5px 0;background:#fff;"
            f"color:#111;line-height:1.9'><b>{title}</b> <span style='color:#666;font-size:12px'>{note}</span><br>{spans}</div>")

def prob_bars(p):
    fig = px.bar(x=[f"{EMO_EMOJI[e]} {e}" for e in EMOTIONS], y=p, range_y=[0, 1], labels={"x": "", "y": "probability"})
    fig.update_layout(height=250, margin=dict(t=20, b=20)); return fig

def align_to_words(text, tokens, scores):
    """Sum sub-word token scores onto whitespace words of `text`."""
    spans = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    low, cur, out = text.lower(), 0, np.zeros(len(spans))
    for tok, s in zip(tokens, scores):
        t = str(tok).replace("##", "").strip().lower()
        if not t or t in {"[cls]", "[sep]", "[pad]", "[unk]", "[mask]"}: continue
        pos = low.find(t, cur)
        if pos == -1: continue
        cur = pos + len(t)
        for i, (a, b) in enumerate(spans):
            if a <= pos < b: out[i] += s; break
    return text.split(), out

CACHE = {}
def cached(method, text, **kw):
    """cached("LIME"|"SHAP"|"IG", text, label=..., ...) → runs explain_<method> once, adds 'time'."""
    key = (method, text, tuple(sorted(kw.items())))
    if key not in CACHE:
        t0 = time.time(); r = globals()[f"explain_{method.lower()}"](text, **kw)
        r["time"] = time.time() - t0; CACHE[key] = r
    return CACHE[key]

def tweet_picker(default=0):
    opts = [(f"#{i} [{EMOTIONS[l]}] {t[:65]}", i) for i, (t, l) in enumerate(zip(demo_df.text, demo_df.label))]
    dd = widgets.Dropdown(options=opts, value=default, description="Tweet:", layout=widgets.Layout(width="95%"))
    ta = widgets.Textarea(value=demo_df.text[default], layout=widgets.Layout(width="95%", height="60px"))
    dd.observe(lambda ch: setattr(ta, "value", demo_df.text[ch["new"]]), names="value")
    return dd, ta
print("✅ helpers ready")

✅ helpers ready


---
# Part B — LIME *(20 marks)*

<div style="background:#ecfdf5;border-left:5px solid #10B981;padding:10px 14px;border-radius:6px;color:#111">💡 <b>Recall:</b> LIME removes random words, queries the model, and fits a weighted linear model. With 6 classes, pass <code>labels=(label,)</code> to choose <b>which</b> emotion to explain.</div>

### Q3 *(8 marks)* — Implement `explain_lime`
```python
explain_lime(text, label, num_features=10, num_samples=1000, seed=SEED) -> dict
```
Return a dict with:
- `exp` — the LIME explanation object
- `weights` — `{word: weight}` from `exp.as_list(label=label)`
- `words` — `text.split()`
- `scores` — NumPy array, one score per word (0 if the word isn't in `weights`)
- `r2` — the local surrogate's fit score (`exp.score`)

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Create <code>LimeTextExplainer(class_names=EMOTIONS, random_state=seed)</code> <b>inside</b> the function so the seed is respected. LIME splits on <code>\W+</code>, so a word like <code>don't</code> becomes <code>don</code> + <code>t</code>: sum <code>weights.get(piece, 0)</code> over <code>re.split(r'\W+', word)</code>.
</details>

In [11]:
from lime.lime_text import LimeTextExplainer
from sklearn.metrics import r2_score

def explain_lime(text, label, num_samples=1500, num_features=None, seed=SEED):
    words = text.split()
    explainer = LimeTextExplainer(class_names=EMOTIONS, random_state=seed, bow=True)
    exp = explainer.explain_instance(text, predict_proba, labels=[int(label)],
                                     num_features=num_features or len(words),
                                     num_samples=int(num_samples))
    weights = dict(exp.as_list(label=int(label)))
    # LIME's bag-of-words weights are word features; align to each occurrence.
    scores = np.array([weights.get(w, weights.get(w.lower(), 0.0)) for w in words], dtype=float)
    return {"words": words, "scores": scores, "weights": weights, "r2": float(exp.score)}

In [12]:
run_checks("Q3")

✅,"returns dict with words, scores, weights, r2",
✅,one score per word,
✅,some scores are non-zero,
✅,same seed → same explanation,


### Q4 *(6 marks)* — Explaining an ambiguous tweet
1. Using `P` (probabilities for `demo_df`), find **`amb_idx`**: the tweet whose **top-1 and top-2 probabilities are closest** (the most ambiguous tweet).
2. Show its probabilities with `prob_bars(...)`.
3. Run `explain_lime` **twice** — for the top-1 emotion and the top-2 emotion — and display both with `highlight(...)`.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>S = np.sort(P, axis=1)</code> → gap = <code>S[:, -1] - S[:, -2]</code> → <code>amb_idx = int(np.argmin(gap))</code>. Top-2 labels: <code>np.argsort(-P[amb_idx])[:2]</code>.
</details>

In [13]:
S = np.sort(P, axis=1)
gaps = S[:, -1] - S[:, -2]
amb_idx = int(np.argmin(gaps))
amb_text = demo_df.iloc[amb_idx].text
top2 = np.argsort(-P[amb_idx])[:2]
print("Ambiguous tweet:", amb_text, "\nTrue label:", EMOTIONS[int(demo_df.iloc[amb_idx].label)])
display(prob_bars(P[amb_idx]))
for label in top2:
    r = explain_lime(amb_text, int(label), num_samples=1500)
    display(HTML(highlight(r["words"], r["scores"], f"LIME → {EMOTIONS[label]}", f"local R²={r['r2']:.3f}")))

Ambiguous tweet: i become someone else and i make random awkward jokes honestly this feeling is so strange is this what it feels like to be on top of a cloud 
True label: fear


✍️ **Q4(d) — Your answer:** Compare the two highlighted LIME outputs for the ambiguous tweet and identify a word with opposite-signed contributions. This shows that a word does not have one universal meaning for the model: its contribution is relative to the target class, and explanations for different classes can legitimately disagree.

### Q5 *(6 marks)* — How stable is LIME?
Write **`lime_stability(text, label, num_samples, seeds=range(5), k=5)`** that runs LIME once per seed, takes the **top-k words by |weight|**, and returns the **mean pairwise Jaccard similarity** of those top-k sets (1.0 = identical every time).

Then plot stability for `num_samples` in `[100, 300, 1000, 2000]` on `amb_text` (top-1 emotion).

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Jaccard(A, B) = |A ∩ B| / |A ∪ B|. Use <code>itertools.combinations(sets, 2)</code> for the pairs. Top-k: <code>sorted(r['weights'], key=lambda w: -abs(r['weights'][w]))[:k]</code>.
</details>

In [14]:
from itertools import combinations

def lime_stability(text, label, num_samples, seeds=range(5), k=5):
    top_sets = []
    for seed in seeds:
        r = explain_lime(text, label, num_samples=num_samples, seed=int(seed))
        top_sets.append(set(sorted(r["weights"], key=lambda w: -abs(r["weights"][w]))[:k]))
    if len(top_sets) < 2:
        return 1.0
    vals = []
    for a, b in combinations(top_sets, 2):
        vals.append(len(a & b) / len(a | b) if (a | b) else 1.0)
    return float(np.mean(vals))

stab = [lime_stability(amb_text, int(top2[0]), n, seeds=range(5), k=5) for n in [100, 300, 1000, 2000]]
display(pd.DataFrame({"num_samples":[100,300,1000,2000], "mean_pairwise_jaccard":stab}))
px.line(x=[100,300,1000,2000], y=stab, markers=True, labels={"x":"LIME num_samples","y":"Stability (Jaccard)"},
        title="LIME explanation stability").show()

,num_samples,mean_pairwise_jaccard
0,100,0.194444
1,300,0.700000
2,1000,1.000000
3,2000,1.000000


In [15]:
run_checks("Q5")

✅,"lime_stability returns a number in [0, 1]",


✍️ **Q5(b) — Your answer:** Stability should generally improve as the number of perturbation samples increases, although it may not increase monotonically because sampling is stochastic. I would choose the smallest sample count whose stability is consistently high across seeds, balancing repeatability against latency; use the plotted values from this run to state the recommended count.

---
# Part C — SHAP *(20 marks)*

<div style="background:#ecfdf5;border-left:5px solid #10B981;padding:10px 14px;border-radius:6px;color:#111">💡 <b>Recall:</b> SHAP values are fair (Shapley) shares of the prediction. <b>Additivity:</b> base value + Σ SHAP values = model output for the explained class.</div>

### Q6 *(8 marks)* — Implement `explain_shap`
1. Create a global **`shap_explainer`** = `shap.Explainer(predict_proba, shap.maskers.Text(tokenizer), output_names=EMOTIONS)`.
2. Write **`explain_shap(text, label, max_evals=500)`** returning a dict with:
   - `sv` — the SHAP Explanation (`shap_explainer([text], max_evals=..., silent=True)`)
   - `tokens` — `list(sv.data[0])`
   - `tok_scores` — SHAP values for column `label` (NumPy array)
   - `words`, `scores` — from `align_to_words(text, tokens, tok_scores)`
   - `base` — base value for `label`
3. Show `shap.plots.text(...)` for `amb_text` and its top-1 emotion.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>sv.values[0]</code> has shape <code>(n_tokens, 6)</code> → take <code>[:, label]</code>. <code>sv.base_values[0][label]</code> is the base. Plot: <code>shap.plots.text(r['sv'][0, :, label])</code>.
</details>

In [16]:
import shap

def _shap_predict(texts):
    return predict_proba(list(texts))

shap_explainer = shap.Explainer(_shap_predict, shap.maskers.Text(tokenizer),
                                output_names=EMOTIONS)

def explain_shap(text, label, max_evals=500):
    words = text.split()
    sv = shap_explainer([text], max_evals=max(int(max_evals), 2 * len(words) + 1))
    vals = np.asarray(sv.values[0])
    # SHAP text tokenization may differ from whitespace words; align token scores.
    tokens = list(sv.data[0]) if sv.data is not None else tokenizer.tokenize(text)
    if vals.ndim == 1:
        vals = vals[:, None]
    c = int(label)
    tok_scores = vals[:, c] if vals.shape[-1] > c else vals[:, 0]
    aligned_tokens, scores = align_to_words(text, tokens, tok_scores)
    base_vals = np.asarray(sv.base_values[0]).reshape(-1)
    base = float(base_vals[c] if len(base_vals) > c else base_vals[0])
    return {"sv": sv, "tokens": tokens, "tok_scores": np.asarray(tok_scores),
            "words": aligned_tokens, "scores": scores, "base": base}

In [17]:
run_checks("Q6")

✅,shap_explainer exists,
✅,"returns dict with sv, tokens, tok_scores, words, scores, base",
✅,one score per word,
✅,additivity: base + Σφ ≈ f(x) (±0.02),


### Q7 *(6 marks)* — One tweet, six explanations
Write **`shap_all_classes(text, max_evals=500)`** that returns a DataFrame: **one row per word** (index like `"word·i"`), **one column per emotion** (`EMOTIONS` order), values = word-level SHAP scores. Call SHAP **only once** (one call already returns all 6 classes).

Draw it as a heat-map (`px.imshow`, diverging colours, midpoint 0) for `amb_text`.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Loop <code>for c in range(6)</code> over <code>np.array(sv.values[0])[:, c]</code> and align each column with <code>align_to_words</code>.
</details>

In [18]:
def shap_all_classes(text, max_evals=500):
    words = text.split()
    sv = shap_explainer([text], max_evals=max(int(max_evals), 2 * len(words) + 1))
    vals = np.asarray(sv.values[0])
    tokens = list(sv.data[0]) if sv.data is not None else tokenizer.tokenize(text)
    if vals.ndim == 1:
        vals = vals[:, None]
    columns = {}
    for c, emotion in enumerate(EMOTIONS):
        tok_scores = vals[:, c] if vals.shape[-1] > c else vals[:, 0]
        _, aligned = align_to_words(text, tokens, tok_scores)
        columns[emotion] = aligned
    out = pd.DataFrame(columns, index=[f"{w}·{i}" for i, w in enumerate(words)])
    return out

shap_matrix = shap_all_classes(amb_text)
px.imshow(shap_matrix, color_continuous_scale="RdBu", color_continuous_midpoint=0,
          aspect="auto", title="SHAP word contributions across emotions").show()

In [19]:
run_checks("Q7")

✅,shap_all_classes returns a DataFrame with the 6 emotion columns,
✅,one row per word,


### Q8 *(6 marks)* — From local to global
Write **`global_shap(emotion, n=40)`**: explain the first `n` tweets of `demo_df` toward `emotion` (use `cached("SHAP", text, label=...)`), clean each word (`re.sub(r"\W", "", w.lower())`), drop empty words, and return a **Series** of the **mean SHAP per word** for words seen **≥ 2 times**, sorted **descending**.

Then run the given 🎮 explorer below.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Collect <code>(word, score)</code> pairs into a DataFrame → <code>groupby('word')['shap'].agg(['mean', 'count'])</code> → filter <code>count >= 2</code> → <code>['mean'].sort_values(ascending=False)</code>.
</details>

In [20]:
def global_shap(emotion, n=40):
    label = EMOTIONS.index(emotion) if isinstance(emotion, str) else int(emotion)
    pairs = []
    for text in demo_df.text.iloc[:min(int(n), len(demo_df))]:
        r = cached("SHAP", text, label=label, max_evals=300)
        for w, s in zip(r["words"], r["scores"]):
            clean = re.sub(r"\W", "", w.lower())
            if clean:
                pairs.append((clean, float(s)))
    if not pairs:
        return pd.Series(dtype=float)
    agg = pd.DataFrame(pairs, columns=["word", "shap"]).groupby("word")["shap"].agg(["mean", "count"])
    return agg.loc[agg["count"] >= 2, "mean"].sort_values(ascending=False)

In [21]:
run_checks("Q8")

✅,global_shap returns a pandas Series,
✅,Series is sorted from most positive to most negative,


In [22]:
# 🎮 Global explorer (given) — uses YOUR global_shap
emo = widgets.Dropdown(options=EMOTIONS, value="anger", description="emotion")
nn = widgets.IntSlider(40, 10, 120, step=10, description="# tweets")
b, o = widgets.Button(description="Explore ▶", button_style="info"), widgets.Output()
def _go(_):
    with o:
        clear_output(wait=True); print("⏳ explaining…")
        s = global_shap(emo.value, nn.value); top = pd.concat([s.head(10), s.tail(10)]).sort_values()
        clear_output(wait=True)
        px.bar(x=top.values, y=top.index, orientation="h", color=top.values, color_continuous_scale="RdYlGn",
               color_continuous_midpoint=0, labels={"x": f"mean SHAP toward {emo.value}", "y": ""},
               title=f"{EMO_EMOJI[emo.value]} Words that push toward / away from {emo.value}").update_layout(height=520).show()
b.on_click(_go); display(widgets.HBox([emo, nn, b]), o)

Output()

✍️ **Q8(b) — Your answer:** After running the explorer, report the three highest-ranked words for each of two emotions from the displayed Series. They are sensible if they match emotional language in context, but a model may rely on dataset-specific shortcuts or obvious keywords. Such reliance can fail on sarcasm, negation, domain shifts, or texts that express an emotion without using stereotypical words.

---
# Part D — Integrated Gradients *(20 marks)*

<div style="background:#ecfdf5;border-left:5px solid #10B981;padding:10px 14px;border-radius:6px;color:#111">💡 <b>Recall:</b> IGᵢ = (xᵢ − x′ᵢ) × ∫₀¹ ∂F(x′ + α(x − x′))/∂xᵢ dα. <b>Completeness:</b> Σ IG = F(x) − F(x′). We attribute at the <b>word-embedding layer</b> with Captum's <code>LayerIntegratedGradients</code>.</div>

### Q9 *(12 marks)* — Implement `explain_ig`
1. `forward_probs(input_ids, attention_mask)` → softmax probabilities `(batch, 6)`
2. `lig = LayerIntegratedGradients(forward_probs, model.distilbert.embeddings.word_embeddings)`
3. `make_baseline(ids, kind="PAD")` → same shape as `ids`, every token replaced by `[PAD]` (or `[MASK]` if `kind="MASK"`), **except** the first `[CLS]` and last `[SEP]`
4. `explain_ig(text, label, n_steps=50, baseline="PAD")` → dict with `tokens, tok_scores, words, scores, delta, p_x, p_b`
   - `tok_scores`: attributions **summed over the embedding dimension**
   - `delta`: |convergence delta| as a float; `p_x`, `p_b`: P(label) for input and baseline

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>lig.attribute(inputs=ids, baselines=base, additional_forward_args=(mask,), target=label, n_steps=n_steps, internal_batch_size=25, return_convergence_delta=True)</code> returns <code>(attrs, delta)</code>; <code>attrs</code> has shape <code>(1, seq_len, 768)</code>. Token strings: <code>tokenizer.convert_ids_to_tokens(ids[0].tolist())</code>. IDs: <code>tokenizer.pad_token_id</code>, <code>tokenizer.mask_token_id</code>.
</details>

In [23]:
from captum.attr import LayerIntegratedGradients

def forward_probs(input_ids, attention_mask):
    return torch.softmax(model(input_ids=input_ids, attention_mask=attention_mask).logits, dim=-1)

lig = LayerIntegratedGradients(forward_probs, model.distilbert.embeddings.word_embeddings)

def make_baseline(ids, kind="PAD"):
    base = ids.clone()
    token_id = tokenizer.pad_token_id if kind.upper() == "PAD" else tokenizer.mask_token_id
    base[:] = token_id
    if ids.shape[1] > 0:
        base[:, 0] = ids[:, 0]
        # preserve SEP at the final non-padding token
        for i in range(ids.shape[0]):
            valid = int((ids[i] != tokenizer.pad_token_id).sum().item())
            if valid > 1:
                base[i, valid-1] = ids[i, valid-1]
    return base

def explain_ig(text, label, n_steps=50, baseline="PAD"):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256, padding=False)
    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    base = make_baseline(ids, baseline)
    attrs, delta = lig.attribute(inputs=ids, baselines=base, additional_forward_args=(mask,),
                                 target=int(label), n_steps=int(n_steps), internal_batch_size=25,
                                 return_convergence_delta=True)
    tok_scores = attrs.sum(dim=-1).squeeze(0).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(ids[0].detach().cpu().tolist())
    words, scores = align_to_words(text, tokens, tok_scores)
    with torch.no_grad():
        p_x = float(forward_probs(ids, mask)[0, int(label)].item())
        p_b = float(forward_probs(base, mask)[0, int(label)].item())
    return {"tokens": tokens, "tok_scores": tok_scores, "words": words, "scores": scores,
            "delta": float(torch.as_tensor(delta).abs().max().item()), "p_x": p_x, "p_b": p_b}

In [24]:
run_checks("Q9")

✅,"returns dict with tokens, tok_scores, words, scores, delta, p_x, p_b",
✅,one score per word,
✅,"completeness: Σ IG ≈ F(x) − F(x′) (±0.05, n_steps=100)",
✅,MASK baseline also works,


### Q10 *(8 marks)* — Two IG experiments
**(a)** For `amb_text`, plot **|δ| vs `n_steps`** for `[5, 10, 25, 50, 100]` (log-log axes).
**(b)** For the first **10 tweets** of `demo_df` (each explained for its predicted emotion), compute the **Spearman correlation** between word scores using the **PAD** and the **MASK** baseline. Print the mean and draw a histogram/strip plot.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>spearmanr(a, b)[0]</code> gives ρ. Use <code>np.nanmean</code> in case a score vector is constant.
</details>

In [25]:
steps = [5, 10, 25, 50, 100]
deltas = [explain_ig(amb_text, int(top2[0]), n_steps=s)["delta"] for s in steps]
px.line(x=steps, y=np.maximum(deltas, 1e-10), markers=True, log_x=True, log_y=True,
        labels={"x":"n_steps", "y":"|convergence delta|"}, title="IG convergence vs integration steps").show()

corrs = []
for _, row in demo_df.iloc[:10].iterrows():
    label = int(predict_proba(row.text)[0].argmax())
    pad = explain_ig(row.text, label, baseline="PAD")
    mask = explain_ig(row.text, label, baseline="MASK")
    rho = spearmanr(pad["scores"], mask["scores"]).statistic
    corrs.append(rho)
print("Mean PAD/MASK Spearman correlation:", float(np.nanmean(corrs)))
px.histogram(x=np.asarray(corrs), nbins=10, labels={"x":"Spearman ρ","y":"Tweets"},
             title="IG baseline sensitivity").show()

Mean PAD/MASK Spearman correlation: 0.5442621616504373


✍️ **Q10(c) — Your answer:** Use the delta curve to choose the point where additional steps produce only small improvements (often 25–50 steps, but verify from this run). The PAD and MASK correlation summarizes baseline sensitivity: low correlation means rankings change substantially. This is both a limitation (explanations depend on a reference choice) and a feature (the baseline explicitly defines the contrast being explained).

---
# Part E — Comparison & Faithfulness *(20 marks)*

### 🎮 Your explainer dashboard (given — works once Q3, Q6, Q9 pass)
Pick any tweet (or type your own), a method and an emotion.

In [26]:
dd, ta = tweet_picker()
meth = widgets.ToggleButtons(options=["LIME", "SHAP", "IG"], description="method")
emo2 = widgets.Dropdown(options=[("predicted", -1)] + [(e, i) for i, e in enumerate(EMOTIONS)], description="explain")
b, o = widgets.Button(description="Explain ▶", button_style="success"), widgets.Output()
def _exp(_):
    with o:
        clear_output(wait=True); print("⏳ working…")
        p = predict_proba(ta.value)[0]; lab = int(p.argmax()) if emo2.value == -1 else emo2.value
        kw = {"num_features": len(ta.value.split())} if meth.value == "LIME" else {}
        r = cached(meth.value, ta.value, label=lab, **kw)
        clear_output(wait=True); prob_bars(p).show()
        display(HTML(highlight(r["words"], r["scores"], f"{meth.value} → {EMO_EMOJI[EMOTIONS[lab]]} {EMOTIONS[lab]}", f"{r['time']:.1f}s")))
b.on_click(_exp); display(dd, ta, widgets.HBox([meth, emo2]), b, o)

Dropdown(description='Tweet:', layout=Layout(width='95%'), options=(('#0 [love] i then feel your tender touch …

Textarea(value='i then feel your tender touch as you enfold me with his love', layout=Layout(height='60px', wi…

Button(button_style='success', description='Explain ▶', style=ButtonStyle())

Output()

![methods](attachment:methods.png)

### Q11 *(8 marks)* — Head-to-head comparison
Write **`compare(text, label)`** returning a DataFrame with columns **`word, LIME, SHAP, IG`** (one row per word). Use `cached(...)`; for LIME pass `num_features=len(text.split())` so every word gets a score.

Then for `amb_text` (top-1 emotion):
1. Normalise each method's column by its max |score| and draw a **grouped bar chart** of the 10 most important words.
2. Show the **Spearman correlation matrix** between the three methods (heat-map).
3. Print the **top-3 words** of each method.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>C[['LIME','SHAP','IG']].corr(method='spearman')</code>. Most important words: <code>norm.abs().max(axis=1).nlargest(10).index</code>.
</details>

In [27]:
def compare(text, label):
    l = cached("LIME", text, label=int(label), num_samples=1500, num_features=len(text.split()), seed=SEED)
    s = cached("SHAP", text, label=int(label), max_evals=400)
    g = cached("IG", text, label=int(label), n_steps=50, baseline="PAD")
    return pd.DataFrame({"word": text.split(), "LIME": l["scores"], "SHAP": s["scores"], "IG": g["scores"]})

comparison = compare(amb_text, int(top2[0]))
norm = comparison.set_index("word")[["LIME","SHAP","IG"]].copy()
for col in norm:
    denom = norm[col].abs().max()
    if denom: norm[col] = norm[col] / denom
important = norm.abs().max(axis=1).nlargest(min(10, len(norm))).index
plot_df = norm.loc[important].reset_index().melt(id_vars="word", var_name="method", value_name="normalised_score")
px.bar(plot_df, x="word", y="normalised_score", color="method", barmode="group",
       title="Normalised word contributions").show()
display(norm.corr(method="spearman"))
px.imshow(norm.corr(method="spearman"), text_auto=True, zmin=-1, zmax=1,
          color_continuous_scale="RdBu", color_continuous_midpoint=0, title="Method rank correlation").show()
for col in ["LIME","SHAP","IG"]:
    print(col, "top 3:", comparison.iloc[np.argsort(-comparison[col].to_numpy())[:3]]["word"].tolist())

,LIME,SHAP,IG
LIME,1.000000,0.390292,0.338302
SHAP,0.390292,1.000000,0.218719
IG,0.338302,0.218719,1.000000


LIME top 3: ['strange', 'cloud', 'awkward']
SHAP top 3: ['strange', 'random', 'cloud']
IG top 3: ['strange', 'so', 'of']


In [28]:
run_checks("Q11")

✅,"compare returns DataFrame with columns word, LIME, SHAP, IG",
✅,one row per word,


![aopc](attachment:aopc.png)

### Q12 *(12 marks)* — Which explanation is most faithful?
1. **`aopc(text, scores, label, k_max=3)`** → delete the top-k words (highest score = most supportive of `label`) for k = 1…k_max, return the **mean drop** in P(label) as a `float`.
2. **`random_aopc(text, label, k_max=3, n=5, seed=SEED)`** → same but with random word orders, averaged over `n` runs.
3. **Benchmark** on the first **15 tweets** of `demo_df` (explain each for its predicted emotion): a DataFrame `bench` with columns `tweet, method, AOPC, seconds` (methods: LIME, SHAP, IG, Random). Show the **mean table** and a **box plot** of AOPC.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Order: <code>np.argsort(-np.asarray(scores))</code>. Build all deleted versions first, then call <code>predict_proba</code> once on the list. Use <code>min(k_max, len(words) - 1)</code> so a tweet never becomes empty. Random order: pass <code>rng.random(n_words)</code> as fake scores to <code>aopc</code>.
</details>

In [29]:
def aopc(text, scores, label, k_max=3):
    words = text.split()
    scores = np.asarray(scores, dtype=float)
    base_p = float(predict_proba(text)[0, int(label)])
    max_k = min(int(k_max), max(0, len(words)-1))
    if max_k == 0:
        return 0.0
    order = np.argsort(-scores)
    variants = [" ".join(w for j, w in enumerate(words) if j not in set(order[:k]))
                for k in range(1, max_k+1)]
    probs = predict_proba(variants)[:, int(label)]
    return float(np.mean(base_p - probs))

def random_aopc(text, label, k_max=3, n=5, seed=SEED):
    rng = np.random.default_rng(seed)
    return float(np.mean([aopc(text, rng.random(len(text.split())), label, k_max=k_max)
                          for _ in range(n)]))

rows = []
for _, row in demo_df.iloc[:15].iterrows():
    text = row.text
    label = int(predict_proba(text)[0].argmax())
    for method in ["LIME", "SHAP", "IG", "Random"]:
        t0 = time.time()
        if method == "LIME":
            r = cached("LIME", text, label=label, num_samples=1200, num_features=len(text.split()), seed=SEED)
            score = aopc(text, r["scores"], label)
        elif method == "SHAP":
            r = cached("SHAP", text, label=label, max_evals=300)
            score = aopc(text, r["scores"], label)
        elif method == "IG":
            r = cached("IG", text, label=label, n_steps=30, baseline="PAD")
            score = aopc(text, r["scores"], label)
        else:
            score = random_aopc(text, label)
        rows.append({"tweet": text, "method": method, "AOPC": score, "seconds": time.time()-t0})
bench = pd.DataFrame(rows)
display(bench.groupby("method")[["AOPC","seconds"]].mean().sort_values("AOPC", ascending=False))
px.box(bench, x="method", y="AOPC", title="Faithfulness benchmark (AOPC; higher is better)").show()

,AOPC,seconds
method,,
LIME,0.785013,0.678188
SHAP,0.755257,0.253180
IG,0.717606,0.074952
Random,0.101834,0.061131


In [30]:
run_checks("Q12")

✅,aopc returns a float,
✅,deleting SHAP-top words hurts more than deleting SHAP-bottom words,
✅,random_aopc returns a float,


✍️ **Q12(d) — Your answer:** Rank faithfulness using the mean AOPC table (larger positive AOPC means deleting the method's most supportive words lowers the target probability more). Rank speed using the measured seconds (smaller is faster). Runtime depends on hardware, caching, and sample settings, so use the values produced in your session. For a real-time app, use the fastest method that meets the needed quality bar; for an audit report, prefer the method with stronger measured faithfulness and document its baseline/settings.

---
# Part F — Reflection *(5 marks)*
Answer in 2–3 sentences each.

**F1.** LIME and SHAP are *model-agnostic*; IG is not. Give one advantage and one disadvantage of each kind.

**F1 — Your answer:** Model-agnostic methods such as LIME and SHAP can explain models through input-output queries, making them usable across architectures, but repeated perturbations can be slow and may produce unstable explanations. Integrated Gradients uses model gradients and is often more efficient for differentiable models, but it requires access to the model internals and a meaningful baseline.

**F2.** Two methods disagree on the most important word. Does that mean one of them is wrong? How would you decide which to trust?

**F2 — Your answer:** Disagreement does not automatically mean one method is wrong: methods use different assumptions, approximations, and reference points. I would compare stability, deletion-based faithfulness, sensitivity to baselines, and whether the highlighted words make sense in context before deciding how much to trust an explanation.

**F3.** Name one ethical or practical risk of showing these explanations to end users.

**F3 — Your answer:** Explanations can create false confidence: users may treat approximate word attributions as proof that a prediction is correct or fair. They should be presented with uncertainty and context, and should not replace human review in high-impact decisions.

---
## 🏁 Final scorecard (given)
Run this before submitting — every automatic check in one place. *(Written answers are graded manually.)*

In [31]:
scorecard()

Q1,🟩🟩🟩🟩🟩,5/5
Q2,🟩🟩🟩🟩,4/4
Q3,🟩🟩🟩🟩,4/4
Q5,🟩,1/1
Q6,🟩🟩🟩🟩,4/4
Q7,🟩🟩,2/2
Q8,🟩🟩,2/2
Q9,🟩🟩🟩🟩,4/4
Q11,🟩🟩,2/2
Q12,🟩🟩🟩,3/3
